# 02a - Data collection

**Aim:** collect positive and negative examples of eukaryotic proteins for signal-peptide prediction.

We used UniProtKB to collect proteins with experimentally supported signal peptides and proteins without an annotated signal peptide. We applied the selection criteria in slide 32 and saved the sequences and metadata as FASTA and TSV files.

**Materials**
- `02a-DataCollection-2026.pdf` (slides 18-19 and 25-32)
- `02a-1-PythonGenerators.ipynb`
- `02a-3-UniProtAPI-examples.ipynb`

## 1. Queries and connection

### 1.1 Imports

In [21]:
import requests
from requests.adapters import HTTPAdapter, Retry
import json
import re
from pathlib import Path

### 1.2 Connection settings

We retrieve JSON entries in batches of 500 and allow up to five retries for temporary server errors (slides 9-10 and 14).

In [22]:
# Retry temporary server errors, as in the course example.
retries = Retry(total=5, backoff_factor=0.25, status_forcelist=[500, 502, 503, 504])
session = requests.Session()
session.mount("https://", HTTPAdapter(max_retries=retries))

uniprot_url = "https://rest.uniprot.org/uniprotkb/search"

### 1.3 Search queries

Both datasets share the following criteria (slide 32):

| Criterion | Query field |
| --- | --- |
| Reviewed entries | `reviewed:true` |
| No fragments | `fragment:false` |
| Eukaryotic proteins | `taxonomy_id:2759` |
| Evidence at protein level | `existence:1` |
| Protein length of at least 40 residues | `length:[40 TO *]` |

For the **positive set**, we require experimental signal-peptide evidence (`ft_signal_exp:*`). We check the cleavage position and SP length after retrieving the entries.

For the **negative set**, we exclude all signal-peptide annotations (`NOT (ft_signal:*)`) and require experimental localization evidence through `cc_scl_term_exp` for at least one of these compartments (slide 27):

| Compartment | UniProt location ID |
| --- | --- |
| Cytosol | `SL-0091` |
| Mitochondrion | `SL-0173` |
| Nucleus | `SL-0191` |
| Plastid | `SL-0209` |
| Peroxisome | `SL-0204` |
| Cell membrane | `SL-0039` |

The queries are encoded into search URLs requesting JSON responses in batches of 500 entries.

In [23]:
positive_query = """
(reviewed:true) AND
(fragment:false) AND
(taxonomy_id:2759) AND
(existence:1) AND
(length:[40 TO *]) AND
(ft_signal_exp:*)
""".replace("\n", " ")


negative_query = """
(reviewed:true) AND
(fragment:false) AND
(taxonomy_id:2759) AND
(existence:1) AND
(length:[40 TO *]) NOT
(ft_signal:*) AND
(
    cc_scl_term_exp:SL-0091 OR
    cc_scl_term_exp:SL-0173 OR
    cc_scl_term_exp:SL-0191 OR
    cc_scl_term_exp:SL-0209 OR
    cc_scl_term_exp:SL-0204 OR
    cc_scl_term_exp:SL-0039
)
""".replace("\n", " ")

# Build the search URLs from the queries, keeping the criteria readable above.
batch_size = 500
positive_url = requests.Request("GET", uniprot_url, params={
    "query": positive_query,
    "format": "json",
    "compressed": "false",
    "size": batch_size,
}).prepare().url

negative_url = requests.Request("GET", uniprot_url, params={
    "query": negative_query,
    "format": "json",
    "compressed": "false",
    "size": batch_size,
}).prepare().url

## 2. Retrieval and filtering

Once the query criteria have been applied by UniProt, we inspect individual features to select valid SP annotations and flag N-terminal transmembrane regions. Separate functions handle retrieval, filtering, and field extraction.

### 2.1 Pagination

`get_batch` yields one response at a time and follows the next-page URL in the `Link` header until all results have been retrieved (slides 14-15). Each request has a 60-second timeout and raises an error if the HTTP response is unsuccessful.

In [24]:
def get_next_link(headers):
    if "Link" in headers:
        # Extract the next-page URL from the Link header.
        match = re.search(r'<([^>]+)>;\s*rel="next"', headers["Link"])
        if match:
            return match.group(1)
    return None


def get_batch(batch_url):
    while batch_url:
        # Run the API call and check for HTTP errors.
        response = session.get(batch_url, timeout=60)
        response.raise_for_status()

        total = response.headers["x-total-results"]
        yield response, total

        # Stop when there is no next-page link.
        batch_url = get_next_link(response.headers)

### 2.2 Shared metadata

For each protein, we extract the accession, organism name, kingdom, and sequence length (slides 30-31). We assign the kingdom from the taxonomic lineage, using `Metazoa`, `Fungi`, `Viridiplantae`, or `Other`.

We retain the lineage label `Viridiplantae` for the category called `Plants` in the slides.

In [25]:
def get_kingdom(entry):
    lineage = entry.get("organism", {}).get("lineage", [])
    for kingdom in ("Metazoa", "Fungi", "Viridiplantae"):
        if kingdom in lineage:
            return kingdom
    return "Other"


def extract_common_fields(entry):
    # Return the fields shared by the positive and negative TSV files.
    accession = entry["primaryAccession"]
    organism = entry["organism"]["scientificName"]
    kingdom = get_kingdom(entry)
    length = entry["sequence"]["length"]
    return accession, organism, kingdom, length

### 2.3 Signal-peptide selection

We inspect the `Signal` annotations in each entry's `features` list and keep the first feature that meets all four conditions (slides 19, 25-26, and 32):

- Experimental support with evidence code `ECO:0000269`.
- A cleavage position that is present and is not marked `UNKNOWN`.
- No `Not cleaved` description.
- An SP length of at least 14 residues.

Since the annotated SP starts at position 1, we use its end position as both its length and the cleavage position. Entries without a qualifying feature are excluded from the positive set.

In [26]:
def get_valid_signal(entry):
    # Find the first signal peptide that meets all selection criteria.
    for feature in entry.get("features", []):
        if feature.get("type") != "Signal":
            continue
        if "Not cleaved" in feature.get("description", ""):
            continue

        end_info = feature.get("location", {}).get("end", {})
        end = end_info.get("value")
        if end is None or end_info.get("modifier") == "UNKNOWN":
            continue
        if end < 14:
            continue

        # Require experimental evidence for this specific feature.
        for evidence in feature.get("evidences", []):
            if evidence.get("evidenceCode") == "ECO:0000269":
                return feature
    return None

### 2.4 N-terminal transmembrane regions

For negative proteins, we record whether a `Transmembrane` feature starts at or before residue 90 (slide 31). We retain these proteins: their N-terminal TM regions can resemble SPs, making them useful difficult negatives.

In [27]:
def has_tm_in_first_90(entry):
    # Keep TM proteins in the negative set and record this flag.
    for feature in entry.get("features", []):
        if feature.get("type") == "Transmembrane":
            start = feature.get("location", {}).get("start", {}).get("value")
            if start is not None and start <= 90:
                return True
    return False

### 2.5 Dataset-specific fields

The filter functions decide which entries to retain. For each accepted entry, the extraction functions return the shared metadata plus one dataset-specific field (slides 30-31).

| Dataset | Entry-level filter | Additional field |
| --- | --- | --- |
| Positive | Require a valid experimental SP. | `cleavage_pos`: the end position of the accepted SP. |
| Negative | Reject any `Signal` annotation, regardless of evidence level. | `has_tm_in_90`: whether a TM region starts within the first 90 residues. |

In [28]:
def filter_positive_entry(entry):
    return get_valid_signal(entry) is not None


def filter_negative_entry(entry):
    # Exclude signal peptides at any evidence level.
    for feature in entry.get("features", []):
        if feature.get("type") == "Signal":
            return False
    return True


def extract_positive_fields(entry):
    accession, organism, kingdom, length = extract_common_fields(entry)
    signal = get_valid_signal(entry)
    cleavage_pos = signal["location"]["end"]["value"]
    return accession, organism, kingdom, length, cleavage_pos


def extract_negative_fields(entry):
    accession, organism, kingdom, length = extract_common_fields(entry)
    has_tm_in_90 = has_tm_in_first_90(entry)
    return accession, organism, kingdom, length, has_tm_in_90

### 2.6 Dataset export

`get_dataset` processes each batch, applies the relevant filter, and writes every accepted protein to both output files: its metadata to the TSV and its full sequence to the FASTA. Entries are written directly to disk, without accumulating the full dataset in memory.

The retrieved and retained counts let us track how many proteins pass the additional filters.

In [29]:
def get_dataset(search_url, filter_function, extract_function, columns, output_prefix):
    n_total, n_filtered = 0, 0
    tsv_path = f"{output_prefix}.tsv"
    fasta_path = f"{output_prefix}.fasta"

    with open(tsv_path, "w", encoding="utf-8") as tsv_out, \
         open(fasta_path, "w", encoding="utf-8") as fasta_out:
        print(*columns, sep="\t", file=tsv_out)

        # Retrieve and parse one batch at a time.
        for batch, total in get_batch(search_url):
            batch_json = json.loads(batch.text)
            for entry in batch_json["results"]:
                n_total += 1
                if filter_function(entry):
                    n_filtered += 1

                    # Extract the fields and write one TSV row.
                    fields = extract_function(entry)
                    print(*fields, sep="\t", file=tsv_out)

                    # Write the same protein to the FASTA file.
                    accession = entry["primaryAccession"]
                    sequence = entry["sequence"]["value"]
                    print(f">{accession}", sequence, sep="\n", file=fasta_out)

    print(f"Done! Total retrieved: {n_total}, passed filter: {n_filtered}")
    print(f"TSV saved to: {tsv_path}")
    print(f"FASTA saved to: {fasta_path}")

## 3. Preliminary datasets

We generate the positive and negative datasets in `data/collected/`, using the output formats described on slides 29-32.

| Dataset | Metadata | Protein sequences |
| --- | --- | --- |
| Positive | `positive_dataset.tsv` | `positive_dataset.fasta` |
| Negative | `negative_dataset.tsv` | `negative_dataset.fasta` |

A new collection run replaces these files; the contents may change with UniProt updates.

In [30]:
output_dir = Path("../data/collected")
output_dir.mkdir(parents=True, exist_ok=True)

positive_columns = ["accession", "organism", "kingdom", "length", "cleavage_pos"]
negative_columns = ["accession", "organism", "kingdom", "length", "has_tm_in_90"]

# Positive dataset: proteins with a valid experimental signal peptide.
get_dataset(
    positive_url, filter_positive_entry, extract_positive_fields,
    positive_columns, output_dir / "positive_dataset"
)

# Negative dataset: proteins without any signal peptide annotation.
get_dataset(
    negative_url, filter_negative_entry, extract_negative_fields,
    negative_columns, output_dir / "negative_dataset"
)

Done! Total retrieved: 2972, passed filter: 2957
TSV saved to: ..\data\collected\positive_dataset.tsv
FASTA saved to: ..\data\collected\positive_dataset.fasta
Done! Total retrieved: 20974, passed filter: 20974
TSV saved to: ..\data\collected\negative_dataset.tsv
FASTA saved to: ..\data\collected\negative_dataset.fasta
